# EMNIST Letters

### Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

### configure parameters, device and manual seed

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 64
lr = 1e-3
epochs = 10
seed = 42
torch.manual_seed(seed)

Using device: cpu


### Data pipeline

- EMNIST letters are rotated by default

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert to tensor
    transforms.Lambda(lambda x: torch.rot90(x, 1, (1, 2)).flip(1)),  # Rotate + flip to upright
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize like MNIST
])

### Load EMNIST database

In [4]:
train_dataset = datasets.EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = datasets.EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)


# Check dataset classes
print(train_dataset.classes)  # should show 26 letters

['N/A', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


### Data Loaders

In [5]:
# Shift labels from 1-26 to 0-25 for PyTorch
train_dataset.targets = train_dataset.targets - 1
test_dataset.targets = test_dataset.targets - 1

print(train_dataset.classes)
# Dataloaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)  # Should be [64, 1, 28, 28], [64]

['N/A', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
torch.Size([64, 1, 28, 28]) torch.Size([64])


### Label -> Letter helper

In [6]:
def to_letter(label):
    return chr(ord('a') + label - 1)  # EMNIST letters labels are 1-26 for a-z

### Create - Convolution Neuarl Network model

In [7]:
class EMNISTModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2), # -> 14 x 14
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2) # -> 7 x 7
    )
    
    self.fc = nn.Sequential(
        nn.Linear(128*7*7, 256),
        nn.ReLU(),
        nn.Dropout(0.25),
        nn.Linear(256, 27)  # 27 classes for EMNIST letters (including background)
    )

  def forward(self, x):
    x = self.conv(x)
    x = torch.flatten(x, 1)
    return self.fc(x)

### Device and Optimizer

In [8]:
model = EMNISTModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

### Training Loop

In [9]:
train_losses = []
test_accuracies = []

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for x, y in train_loader:
        # filter out label 0 if exists
        mask = y != 0
        x, y = x[mask], y[mask]
        y = (y - 1).to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()

    train_loss = running_loss / total_train
    train_losses.append(train_loss)

    model.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for x, y in test_loader:
            # filter out label 0 if exists
            mask = y != 0
            x, y = x[mask], y[mask]
            y = (y - 1).to(device)
            preds = model(x).argmax(1)
            correct_test += (preds == y).sum().item()
            total_test += y.size(0)

    test_accuracy = correct_test / total_test
    test_accuracies.append(test_accuracy)

    print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/10, Train Loss: 0.3623, Test Accuracy: 0.9359
Epoch 2/10, Train Loss: 0.1922, Test Accuracy: 0.9398
Epoch 3/10, Train Loss: 0.1629, Test Accuracy: 0.9411
Epoch 4/10, Train Loss: 0.1386, Test Accuracy: 0.9436
Epoch 5/10, Train Loss: 0.1244, Test Accuracy: 0.9435
Epoch 6/10, Train Loss: 0.1092, Test Accuracy: 0.9436
Epoch 7/10, Train Loss: 0.0994, Test Accuracy: 0.9439
Epoch 8/10, Train Loss: 0.0913, Test Accuracy: 0.9447
Epoch 9/10, Train Loss: 0.0847, Test Accuracy: 0.9456
Epoch 10/10, Train Loss: 0.0777, Test Accuracy: 0.9416
